# Task 6 - Idempotent replay verification

The replay run is captured as four ordered stages from one automation script.

| Stage | Hash | File nodes | File edges | Total nodes | Total edges | Mongo docs | Mongo offset | Checkpoint | Result |
|---|---|---:|---:|---:|---:|---:|---:|---:|---|
| Locked baseline | `ac342f9e8fd3` | 7 | 6 | 62375 | 77819 | 61 | 130 | 132 | PASS |
| Modified file | `079eca803ea5` | 29 | 36 | 62397 | 77849 | 61 | 132 | 134 | PASS |
| Forced unchanged replay | `079eca803ea5` | 29 | 36 | 62397 | 77849 | 61 | 134 | 136 | PASS |
| Spark restart + replay | `079eca803ea5` | 29 | 36 | 62397 | 77849 | 61 | 136 | 138 | PASS |

The script temporarily restores the locked file, restores the modified bytes in a `finally` block, polls both databases, and records assertions only after every sink converges.

In [1]:
import json
from pathlib import Path

root = Path('..').resolve()
evidence = json.loads((root / 'evidence/runtime/verification.json').read_text(encoding='utf-8'))
summary = {
    'repository': evidence['repository'],
    'replay_file': {k: v for k, v in evidence['replay_file'].items() if k != 'git_diff'},
    'stages': {
        name: {
            'source_hash': stage['source_hash'],
            'parser': stage['parser'],
            'neo4j': stage['neo4j'],
            'mongo': stage['mongo'],
            'kafka_metadata_end_offset': stage['kafka_metadata_end_offset'],
            'spark_checkpoint_offset': stage['spark_checkpoint_offset'],
        }
        for name, stage in evidence['stages'].items()
    },
    'spark_restart': evidence['spark_restart'],
    'neo4j_dlq_end_offset': evidence['neo4j_dlq_end_offset'],
    'assertions': evidence['assertions'],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
assert evidence['assertions'] and all(evidence['assertions'].values())
print('PASS: every captured replay assertion is true')

{
  "repository": {
    "repo_id": "huggingface/optimum",
    "url": "https://github.com/huggingface/optimum.git",
    "commit_sha": "a6c775e11118d62712057bd3a8c5649898a5312d",
    "raw_python_files": 74,
    "processed_python_files": 61,
    "baseline_python_lines": 13807,
    "modified_python_lines": 13814,
    "parseable_files": 61,
    "parse_success_rate": 1.0
  },
  "replay_file": {
    "path": "optimum/version.py",
    "file_id": "867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7",
    "backup_path": "tmp\\replay-backups\\version-20260722-080605.py.bak"
  },
  "stages": {
    "baseline": {
      "source_hash": "ac342f9e8fd3f74b822e8d7399806be6f17b733b37e1a01ae24a486ba8d330ce",
      "parser": {
        "path": "optimum/version.py",
        "file_id": "867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7",
        "content_hash": "ac342f9e8fd3f74b822e8d7399806be6f17b733b37e1a01ae24a486ba8d330ce",
        "status": "processed",
        "nodes": 7,
      

In [2]:
import subprocess
from pathlib import Path

root = Path('..').resolve()
result = subprocess.run(
    ['git', '-C', str(root / 'source-repo'), 'diff', '--', 'optimum/version.py'],
    capture_output=True, text=True, check=True,
)
print(result.stdout.rstrip())
assert 'lab04_replay_probe' in result.stdout
print('PASS: replay modification is present in exactly optimum/version.py')

diff --git a/optimum/version.py b/optimum/version.py
index bb061c5..a41eacc 100644
--- a/optimum/version.py
+++ b/optimum/version.py
@@ -13,3 +13,10 @@
 #  limitations under the License.
 
 __version__ = "2.2.0.dev0"
+
+
+def lab04_replay_probe(value):
+    """Small deterministic branch used only by the Lab 04 replay demo."""
+    if value > 10:
+        return value * 2
+    return value
PASS: replay modification is present in exactly optimum/version.py


In [3]:
import json
import sys
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root / 'scripts'))
from capture_replay_evidence import checkpoint_offset, kafka_end_offset, mongo_snapshot, neo4j_snapshot

evidence = json.loads((root / 'evidence/runtime/verification.json').read_text(encoding='utf-8'))
expected = evidence['stages']['restart_replay']
live = {
    'neo4j': neo4j_snapshot('867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7'),
    'mongo': mongo_snapshot('867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7'),
    'kafka_metadata_end_offset': kafka_end_offset('cpg.source-metadata.v1'),
    'spark_checkpoint_offset': checkpoint_offset(),
}
print(json.dumps(live, indent=2))
assert live['neo4j'] == expected['neo4j']
assert live['mongo'] == expected['mongo']
assert live['kafka_metadata_end_offset'] == expected['kafka_metadata_end_offset']
assert live['spark_checkpoint_offset'] == expected['spark_checkpoint_offset']
print('PASS: live final state matches the captured restart-replay stage')

{
  "neo4j": {
    "nodes": 62397,
    "unique_nodes": 62397,
    "edges": 77849,
    "unique_edges": 77849,
    "file_nodes": 29,
    "file_edges": 36,
    "edge_kinds": {
      "AST": 57960,
      "CALL": 2593,
      "CFG": 7248,
      "DFG": 10048
    }
  },
  "mongo": {
    "documents": 61,
    "distinct_files": 61,
    "document": {
      "_id": "867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7",
      "path": "optimum/version.py",
      "content_hash": "079eca803ea5bfe068bc805997b013cc14c9ddc8629a2ec634d12bcebb6720ce",
      "node_counts": {
        "AST": 25,
        "SYNTHETIC": 4
      },
      "edge_counts": {
        "AST": 24,
        "CFG": 9,
        "DFG": 3
      },
      "processed_at": "2026-07-22T01:08:09.918576Z",
      "run_id": "fb81462f6af147b5b74e33a33e1efa72",
      "kafka_offset": 136
    },
    "other_documents_count": 60,
    "other_documents_digest": "8aa80ee9361d8a1e4caf27fc02ca9ed83677a9820e0b2dbe011d53cdaaea0303"
  },
  "kafka_metadata_en

## Reflection

The evidence distinguishes parser idempotency, database uniqueness, unchanged-file stability, and Spark offset recovery. Matching hashes and digests show that only the requested file changed, while final live queries prevent the table from being an unsupported claim.